# Flattened-Hamiltonian twist / entanglement benchmark

This standalone local-CPU notebook diagnoses the near-\(\pi\) entanglement-spectrum recrossing for the \(20\times40\), \(n_{\rm shell}=1\), trial-\(X\), \(\alpha_{\rm top}=1\), \(\alpha_{\rm triv}=30\) domain-wall flattened Hamiltonian.

It compares domain-wall truncation off/on; instantaneous and overlap-continued half-filled states; both twist orientations; uniform and seam gauges; the current half-\(y\) cut and full-\(y\) cylinder cuts; multiple twist meshes, tracked-mode counts, reference levels, wall windows, and endpoint windows. This is an estimator benchmark for exact flattened states—not evidence about monitored trajectories.

The production configuration is intentionally expensive (roughly 10–30 CPU minutes depending on the machine). Set the editable CPU controls below before running later cells. Set the environment variable FLAT_TWIST_SMOKE=1 for a small end-to-end test.

## CPU controls (run before numerical imports)

In [ ]:
import os
import multiprocessing as _mp

# Editable inclusive CPU range. Leave both as None to use CPU_COUNT from the
# process's currently allowed affinity mask.
CPU_START = None
CPU_STOP = None
CPU_COUNT = 8

_allowed = sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else list(range(_mp.cpu_count()))
if CPU_START is not None or CPU_STOP is not None:
    if CPU_START is None or CPU_STOP is None or int(CPU_STOP) < int(CPU_START):
        raise ValueError("Set both CPU_START and CPU_STOP with CPU_STOP >= CPU_START.")
    _requested = list(range(int(CPU_START), int(CPU_STOP) + 1))
    _resolved = [cpu for cpu in _requested if cpu in _allowed]
    if not _resolved:
        raise RuntimeError("The requested CPU range does not intersect the process affinity mask.")
else:
    _resolved = _allowed[:max(1, min(int(CPU_COUNT), len(_allowed)))]

if hasattr(os, "sched_setaffinity"):
    os.sched_setaffinity(0, _resolved)
for _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
              "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_name] = str(len(_resolved))

SMOKE = os.environ.get("FLAT_TWIST_SMOKE", "0") == "1"
OUTPUT_ROOT_OVERRIDE = os.environ.get("FLAT_TWIST_OUTPUT_ROOT", "")
print({"smoke": SMOKE, "resolved_cpus": _resolved, "thread_count": len(_resolved)})

## Imports, repository source, and reproducibility metadata

In [ ]:
from pathlib import Path
import contextlib
import datetime as _dt
import hashlib
import io
import json
import math
import time
import warnings

_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
REPO_ROOT = next((p for p in _candidates if (p / "src" / "fgtn" / "classA_U1FGTN.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run from this repository or one of its subdirectories.")
import sys
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

from fgtn.classA_U1FGTN import classA_U1FGTN

SOURCE_PATH = REPO_ROOT / "src" / "fgtn" / "classA_U1FGTN.py"
SOURCE_HASH = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
RUN_ID = _dt.datetime.now().strftime("run_%Y%m%d_%H%M%S")
_default_output = REPO_ROOT / "notebooks" / "flattened_hamiltonian_analysis" / "outputs" / "flattened_twist_entanglement_benchmark_v1"
OUTPUT_BASE = Path(OUTPUT_ROOT_OVERRIDE).resolve() if OUTPUT_ROOT_OVERRIDE else _default_output
OUTPUT_DIR = OUTPUT_BASE / RUN_ID
SAVE_OUTPUTS = True
FIGURE_DIR = OUTPUT_DIR / "figures"

def save_figure(fig, stem):
    if not SAVE_OUTPUTS:
        return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["CMU Sans Serif", "DejaVu Sans"],
    "mathtext.fontset": "cm",
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})
print({"repo_root": str(REPO_ROOT), "source_sha256": SOURCE_HASH, "output_dir": str(OUTPUT_DIR)})

## Benchmark configuration

The closed loop uses a \(10^{-7}\) offset, so no sampled endpoint is forced onto a zero mode. The 33- and 65-point meshes are exact nested subsamples of the 129-point global mesh. A separate 129-point mesh resolves \(0.70\pi\leq|\phi|\leq1.05\pi\).

In [ ]:
if SMOKE:
    NX, NY = 8, 12
    DW_INTERVAL = (2, 6)
    GLOBAL_N, REFINE_N = 9, 9
    TRACKED_MODES = (4, 6, 8)
    WALL_HALF_WIDTHS = (1, 2)
    ENDPOINT_HALF_WIDTHS = (1, 2)
else:
    NX, NY = 20, 40
    DW_INTERVAL = (5, 15)
    GLOBAL_N, REFINE_N = 129, 129
    TRACKED_MODES = (16, 24, 32, 48)
    WALL_HALF_WIDTHS = (1, 2, 3)
    ENDPOINT_HALF_WIDTHS = (1, 2, 3)

NSHELL = 1
TRIAL_ORBITAL = "X"
ALPHA_TOP, ALPHA_TRIV = 1.0, 30.0
DW_TRUNCATIONS = (False, True)
TWIST_DIRECTIONS = (+1, -1)
PHI_OFFSET = 1e-7
REFERENCE_LEVELS = (-0.05, -0.025, 0.0, 0.025, 0.05)
PRIMARY_TRACKED_MODES = 24 if 24 in TRACKED_MODES else TRACKED_MODES[len(TRACKED_MODES)//2]
PRIMARY_HALF_WIDTH = 2 if 2 in WALL_HALF_WIDTHS else WALL_HALF_WIDTHS[0]
DEGENERACY_TOL = 1e-9
NUMERICAL_TOL = 1e-10
PROJECTOR_CLOSURE_TOL = 5e-9  # endpoint eigensolver precision near wall-level crossings
RANK = NX * NY
ORBITALS_PER_Y = 2 * NX

CONFIG = {
    "Nx": NX, "Ny": NY, "rank": RANK, "nshell": NSHELL,
    "trial_orbital": TRIAL_ORBITAL, "alpha_top": ALPHA_TOP,
    "alpha_triv": ALPHA_TRIV, "dw_interval": DW_INTERVAL,
    "dw_truncations": DW_TRUNCATIONS, "twist_directions": TWIST_DIRECTIONS,
    "global_points": GLOBAL_N, "refined_points": REFINE_N,
    "phi_offset": PHI_OFFSET, "tracked_modes": TRACKED_MODES,
    "reference_levels": REFERENCE_LEVELS,
    "wall_half_widths": WALL_HALF_WIDTHS,
    "endpoint_half_widths": ENDPOINT_HALF_WIDTHS,
    "source_sha256": SOURCE_HASH, "smoke": SMOKE,
}
_builds = len(DW_TRUNCATIONS) * len(TWIST_DIRECTIONS) * (GLOBAL_N + REFINE_N - 1)
_est_seconds = _builds * (0.08 if SMOKE else 1.25)
print(json.dumps(CONFIG, indent=2, default=list))
print(f"Estimated wall time: {_est_seconds/60:.1f} min (machine-dependent; {_builds} twist builds).")

## Canonical flattened Hamiltonian and gauge/cut helpers

The one-particle basis follows the canonical code: \(i=\mu+2x+2N_x y\). The flattened Hamiltonian is built only from the four canonical overcomplete-Wannier frames. No dense Hamiltonian or complete eigenvector history is saved.

In [ ]:
def _frame_projector(frame):
    v = np.asarray(frame, dtype=np.complex128).reshape(2 * NX * NY, NX * NY, order="F")
    return v @ v.conj().T

def build_flattened_hamiltonian(phi, dw_truncation):
    with contextlib.redirect_stdout(io.StringIO()):
        model = classA_U1FGTN(
            Nx=NX, Ny=NY, DW=True, nshell=NSHELL, filling_frac=0.5,
            alpha_1=ALPHA_TOP, alpha_2=ALPHA_TRIV,
            trial_orbitals=TRIAL_ORBITAL, dw_truncation=dw_truncation,
            twist_y=float(phi), dw_interval=DW_INTERVAL,
        )
        model.construct_OW_projectors(
            nshell=NSHELL, DW=True, trial_orbitals=TRIAL_ORBITAL,
            dw_truncation=dw_truncation, twist_y=float(phi),
        )
    h = (_frame_projector(model.WF_Ap) + _frame_projector(model.WF_Bp)
         - _frame_projector(model.WF_Am) - _frame_projector(model.WF_Bm))
    return np.asarray(0.5 * (h + h.conj().T), dtype=np.complex128)

def y_block_decompose(matrix):
    shaped = np.asarray(matrix).reshape(NY, ORBITALS_PER_Y, NY, ORBITALS_PER_Y)
    transformed = np.fft.ifft(np.fft.fft(shaped, axis=0), axis=2)
    blocks = np.stack([transformed[k, :, k, :] for k in range(NY)])
    off = transformed.copy()
    for k in range(NY):
        off[k, :, k, :] = 0
    return blocks, float(np.max(np.abs(off)))

def y_block_reconstruct(blocks):
    transformed = np.zeros((NY, ORBITALS_PER_Y, NY, ORBITALS_PER_Y), dtype=np.complex128)
    for k in range(NY):
        transformed[k, :, k, :] = blocks[k]
    shaped = np.fft.ifft(np.fft.fft(transformed, axis=2), axis=0)
    return shaped.reshape(2 * NX * NY, 2 * NX * NY)

def block_eigh(blocks):
    vals, vecs = [], []
    for block in blocks:
        e, v = np.linalg.eigh(block)
        vals.append(e)
        vecs.append(v)
    return np.asarray(vals), np.asarray(vecs)

def projectors_from_block_occupations(vecs, occupied):
    out = np.empty((NY, ORBITALS_PER_Y, ORBITALS_PER_Y), dtype=np.complex128)
    for k in range(NY):
        vk = vecs[k][:, occupied[k]]
        out[k] = vk @ vk.conj().T
    return out

def instantaneous_occupations(evals, rank=RANK):
    flat_order = np.argsort(evals, axis=None)[:rank]
    occupied = np.zeros_like(evals, dtype=bool)
    occupied[np.unravel_index(flat_order, evals.shape)] = True
    return occupied

def phase_to_seam(phi, seam_y):
    # D maps the uniform-link gauge to a gauge with all flux on bond seam_y -> seam_y+1.
    a = float(phi) / NY
    theta = np.zeros(NY)
    start = (int(seam_y) + 1) % NY
    y = start
    for _ in range(NY - 1):
        yn = (y + 1) % NY
        theta[yn] = theta[y] + a
        y = yn
    return np.repeat(np.exp(1j * theta), ORBITALS_PER_Y)

def large_gauge_phase(direction):
    y = np.repeat(np.arange(NY), ORBITALS_PER_Y)
    return np.exp(1j * float(direction) * 2 * np.pi * y / NY)

def apply_diagonal_gauge(matrix, phase):
    return phase[:, None] * matrix * phase.conj()[None, :]

def circular_distance(values, center, period):
    values = np.asarray(values)
    return np.abs((values - center + period / 2) % period - period / 2)

def site_coordinates(indices):
    indices = np.asarray(indices, dtype=int)
    y = indices // ORBITALS_PER_Y
    rem = indices % ORBITALS_PER_Y
    x = rem // 2
    mu = rem % 2
    return x, y, mu

def indices_for_mask(mask_xy):
    xs, ys = np.where(np.asarray(mask_xy, dtype=bool))
    return np.sort(np.concatenate([2 * xs + 2 * NX * ys, 2 * xs + 1 + 2 * NX * ys]))

def make_cut_specs():
    specs = {}
    half = np.zeros((NX, NY), dtype=bool)
    half[:, :NY // 2] = True
    specs["half_y_current"] = {
        "indices": indices_for_mask(half), "kind": "half_y",
        "description": f"all x, y=0,...,{NY//2-1}",
    }
    x_left, x_right = DW_INTERVAL
    for delta in (0, 1, 2):
        lo, hi = x_left + delta, x_right - delta
        if lo <= hi:
            strip = np.zeros((NX, NY), dtype=bool)
            strip[lo:hi + 1, :] = True
            specs[f"full_y_delta{delta}"] = {
                "indices": indices_for_mask(strip), "kind": "full_y",
                "description": f"x={lo},...,{hi}; all y",
            }
    return specs

CUT_SPECS = make_cut_specs()
print({name: {"size": len(spec["indices"]), **{k:v for k,v in spec.items() if k != "indices"}}
       for name, spec in CUT_SPECS.items()})

## Continuation, entanglement tracking, weights, and crossing estimators

In [ ]:
def polar_transport(reference, target):
    overlap = reference.conj().T @ target
    u, singular_values, vh = np.linalg.svd(overlap, full_matrices=False)
    return target @ (vh.conj().T @ u.conj().T), singular_values

def overlap_continue(previous_vecs, current_evals, current_vecs):
    tracked_evals = np.empty_like(current_evals)
    tracked_vecs = np.empty_like(current_vecs)
    diagnostics = []
    for k in range(NY):
        overlap = np.abs(previous_vecs[k].conj().T @ current_vecs[k]) ** 2
        rows, cols = linear_sum_assignment(-overlap)
        permutation = cols[np.argsort(rows)]
        e = current_evals[k, permutation]
        v = current_vecs[k][:, permutation]
        assigned = np.sqrt(np.maximum(overlap[np.arange(ORBITALS_PER_Y), permutation], 0))
        competing = np.partition(np.sqrt(np.maximum(overlap, 0)), -2, axis=1)[:, -2]
        clusters = []
        unused = set(range(ORBITALS_PER_Y))
        while unused:
            seed = min(unused)
            cluster = {j for j in unused if abs(e[j] - e[seed]) < DEGENERACY_TOL}
            unused -= cluster
            clusters.append(sorted(cluster))
        principal_min = 1.0
        for cluster in clusters:
            idx = np.asarray(cluster)
            aligned, singular = polar_transport(previous_vecs[k][:, idx], v[:, idx])
            v[:, idx] = aligned
            principal_min = min(principal_min, float(singular.min()))
        tracked_evals[k] = e
        tracked_vecs[k] = v
        diagnostics.append({
            "assigned_min": float(assigned.min()),
            "competing_max": float(competing.max()),
            "principal_cos_min": principal_min,
            "selection_gap": float(np.min(np.abs(e[:, None] - e[None, :] + np.eye(len(e)) * 1e9))),
        })
    return tracked_evals, tracked_vecs, diagnostics

def entanglement_eigh(projector, indices):
    c = projector[np.ix_(indices, indices)]
    nu, vecs = np.linalg.eigh(0.5 * (c + c.conj().T))
    nu = np.clip(nu.real, 1e-14, 1 - 1e-14)
    eps = np.log((1 - nu) / nu)
    order = np.argsort(eps)
    return eps[order], nu[order], vecs[:, order]

def mode_weights(vecs, indices, cut_kind):
    x, y, _ = site_coordinates(indices)
    density = np.abs(vecs) ** 2
    wall = {}
    for width in WALL_HALF_WIDTHS:
        wall[width] = np.stack([
            density[circular_distance(x, center, NX) <= width].sum(axis=0)
            for center in DW_INTERVAL
        ], axis=1)
    sectors = {}
    if cut_kind == "half_y":
        endpoint_centers = (0, NY // 2 - 1)
        for wwidth in WALL_HALF_WIDTHS:
            for ewidth in ENDPOINT_HALF_WIDTHS:
                sector = np.empty((vecs.shape[1], 2, 2))
                for iw, wall_center in enumerate(DW_INTERVAL):
                    wm = circular_distance(x, wall_center, NX) <= wwidth
                    for ie, endpoint_center in enumerate(endpoint_centers):
                        em = np.abs(y - endpoint_center) <= ewidth
                        sector[:, iw, ie] = density[wm & em].sum(axis=0)
                sectors[(wwidth, ewidth)] = sector
    return wall, sectors

def select_central_modes(eps, vecs, count):
    count = min(int(count), len(eps))
    pick = np.argsort(np.abs(eps))[:count]
    pick = pick[np.argsort(eps[pick])]
    return eps[pick], vecs[:, pick]

class EntanglementTracker:
    def __init__(self, cut_name, cut_spec, state_family):
        self.cut_name = cut_name
        self.cut_spec = cut_spec
        self.state_family = state_family
        self.previous = {k: None for k in TRACKED_MODES}
        self.records = {k: [] for k in TRACKED_MODES}
        self.inertia = []

    def update(self, phi, projector):
        indices = self.cut_spec["indices"]
        eps_all, nu_all, vecs_all = entanglement_eigh(projector, indices)
        inertia_record = {"phi": float(phi)}
        for ref in REFERENCE_LEVELS:
            inertia_record[f"below_{ref:+.3f}"] = int(np.count_nonzero(eps_all < ref))
        self.inertia.append(inertia_record)
        for count in TRACKED_MODES:
            eps, vecs = select_central_modes(eps_all, vecs_all, count)
            previous = self.previous[count]
            if previous is None:
                assigned = np.ones(len(eps))
                competing = np.zeros(len(eps))
                principal = np.ones(len(eps))
            else:
                overlap = np.abs(previous.conj().T @ vecs)
                rows, cols = linear_sum_assignment(-overlap)
                permutation = cols[np.argsort(rows)]
                eps, vecs = eps[permutation], vecs[:, permutation]
                assigned = overlap[np.arange(len(eps)), permutation]
                competing = np.partition(overlap, -2, axis=1)[:, -2] if len(eps) > 1 else np.zeros(1)
                # SVD is a label-independent subspace diagnostic.  Do not rotate
                # distinct entanglement eigenmodes; only fix their individual phases.
                singular = np.linalg.svd(previous.conj().T @ vecs, compute_uv=False)
                diagonal_overlap = np.diag(previous.conj().T @ vecs)
                phases = np.exp(-1j * np.angle(diagonal_overlap))
                vecs = vecs * phases[None, :]
                principal = np.full(len(eps), singular.min())
            wall, sectors = mode_weights(vecs, indices, self.cut_spec["kind"])
            selection_gap = np.sort(np.abs(eps_all))[len(eps)] - np.sort(np.abs(eps_all))[len(eps)-1] if len(eps) < len(eps_all) else np.inf
            self.records[count].append({
                "phi": float(phi), "eps": eps.copy(),
                "assigned": assigned.copy(), "competing": competing.copy(),
                "principal": principal.copy(), "selection_gap": float(selection_gap),
                "wall": {w: value.copy() for w, value in wall.items()},
                "sectors": {key: value.copy() for key, value in sectors.items()},
            })
            self.previous[count] = vecs.copy()

    def arrays(self, count):
        records = self.records[count]
        return {
            "phi": np.array([r["phi"] for r in records]),
            "eps": np.stack([r["eps"] for r in records]),
            "assigned": np.stack([r["assigned"] for r in records]),
            "competing": np.stack([r["competing"] for r in records]),
            "principal": np.stack([r["principal"] for r in records]),
            "selection_gap": np.array([r["selection_gap"] for r in records]),
            "wall": {w: np.stack([r["wall"][w] for r in records]) for w in WALL_HALF_WIDTHS},
            "sectors": {
                key: np.stack([r["sectors"][key] for r in records])
                for key in records[0]["sectors"]
            } if records and records[0]["sectors"] else {},
        }

def signed_crossings(phi, eps, weights, reference=0.0, ambiguity_tol=0.05):
    rows = []
    shifted = eps - float(reference)
    for i in range(len(phi) - 1):
        for mode in range(eps.shape[1]):
            a, b = shifted[i, mode], shifted[i + 1, mode]
            if a == 0 or b == 0 or a * b < 0:
                slope = np.sign(b - a)
                mean_weight = 0.5 * (weights[i, mode] + weights[i + 1, mode])
                winner = int(np.argmax(mean_weight))
                order = np.sort(mean_weight)
                ambiguous = len(order) > 1 and order[-1] - order[-2] < ambiguity_tol
                rows.append({
                    "i0": i, "i1": i + 1, "mode": mode,
                    "phi_mid": float(0.5 * (phi[i] + phi[i + 1])),
                    "signed_flow": int(slope), "sector": winner,
                    "weight": float(mean_weight[winner]),
                    "ambiguous": bool(ambiguous),
                })
    return rows

def inertia_crossings(phi, inertia_records, reference):
    counts = np.array([r[f"below_{reference:+.3f}"] for r in inertia_records])
    rows = []
    for i, delta in enumerate(np.diff(counts)):
        if delta:
            rows.append({
                "i0": i, "i1": i + 1,
                "phi_mid": float(0.5 * (phi[i] + phi[i + 1])),
                "signed_flow": int(-delta),
                "ambiguous": bool(abs(delta) > 1),
            })
    return rows

def make_twist_grids(direction):
    direction = int(direction)
    global_grid = PHI_OFFSET + direction * np.linspace(0, 2 * np.pi, GLOBAL_N)
    refined = PHI_OFFSET + direction * np.linspace(0.70 * np.pi, 1.05 * np.pi, REFINE_N)
    combined = np.unique(np.round(np.concatenate([global_grid, refined]), 15))
    combined = np.sort(combined)
    if direction < 0:
        combined = combined[::-1]
    return global_grid, refined, combined

def nearest_indices(master, target):
    return np.array([int(np.argmin(np.abs(master - value))) for value in target], dtype=int)

def grid_index_sets(master, global_grid, refined):
    out = {
        f"global_{GLOBAL_N}": nearest_indices(master, global_grid),
        f"refined_{REFINE_N}": nearest_indices(master, refined),
        "combined": np.arange(len(master)),
    }
    if GLOBAL_N == 129:
        out["global_65"] = nearest_indices(master, global_grid[::2])
        out["global_33"] = nearest_indices(master, global_grid[::4])
    elif GLOBAL_N >= 9:
        out[f"global_{(GLOBAL_N+1)//2}"] = nearest_indices(master, global_grid[::2])
    return out

## One-case driver and acceptance diagnostics

The continued state fixes the initial block-resolved occupation labels and transports each \(k_y\)-block eigenbasis with Hungarian assignment followed by polar alignment. The instantaneous state reselects the globally lowest fixed rank at every twist.

In [ ]:
def run_case(dw_truncation, direction):
    global_grid, refined, phi_values = make_twist_grids(direction)
    trackers = {
        (family, name): EntanglementTracker(name, spec, family)
        for family in ("instantaneous", "continued")
        for name, spec in CUT_SPECS.items()
    }
    previous_vecs = None
    continued_occupied = None
    first_h = first_p_inst = first_p_cont = None
    physical = {"phi": [], "evals": [], "wall": [], "continuation": []}
    wall_charge = {
        "phi": [], "instantaneous": [], "continued": [],
        "basin_instantaneous": [], "basin_continued": [],
        "density_x_instantaneous": [], "density_x_continued": [],
    }
    full_x = np.tile(np.repeat(np.arange(NX), 2), NY)
    distance_left = circular_distance(full_x, DW_INTERVAL[0], NX)
    distance_right = circular_distance(full_x, DW_INTERVAL[1], NX)
    basin_left = (distance_left < distance_right).astype(float)
    basin_left += 0.5 * (distance_left == distance_right)
    basin_weights = np.stack([basin_left, 1.0 - basin_left])
    validation = {
        "hermiticity_max": 0.0, "block_offdiag_max": 0.0,
        "block_reconstruction_max": 0.0, "projector_idempotency_max": 0.0,
        "rank_error_max": 0.0, "seam_spectrum_max": 0.0,
        "seam_entanglement_max": 0.0,
    }

    for step, phi in enumerate(tqdm(phi_values, desc=f"trunc={dw_truncation}, dir={direction:+d}")):
        h = build_flattened_hamiltonian(phi, dw_truncation)
        validation["hermiticity_max"] = max(validation["hermiticity_max"], float(np.max(np.abs(h - h.conj().T))))
        blocks, offdiag = y_block_decompose(h)
        reconstructed = y_block_reconstruct(blocks)
        validation["block_offdiag_max"] = max(validation["block_offdiag_max"], offdiag)
        validation["block_reconstruction_max"] = max(
            validation["block_reconstruction_max"], float(np.max(np.abs(h - reconstructed)))
        )
        raw_evals, raw_vecs = block_eigh(blocks)

        instant_occ = instantaneous_occupations(raw_evals)
        p_inst_blocks = projectors_from_block_occupations(raw_vecs, instant_occ)
        p_inst = y_block_reconstruct(p_inst_blocks)

        if previous_vecs is None:
            continued_evals, continued_vecs = raw_evals.copy(), raw_vecs.copy()
            continued_occupied = instant_occ.copy()
            continuation_diag = [{"assigned_min": 1.0, "competing_max": 0.0,
                                  "principal_cos_min": 1.0, "selection_gap": np.inf} for _ in range(NY)]
        else:
            continued_evals, continued_vecs, continuation_diag = overlap_continue(
                previous_vecs, raw_evals, raw_vecs
            )
        p_cont_blocks = projectors_from_block_occupations(continued_vecs, continued_occupied)
        p_cont = y_block_reconstruct(p_cont_blocks)
        previous_vecs = continued_vecs.copy()

        for projector in (p_inst, p_cont):
            validation["projector_idempotency_max"] = max(
                validation["projector_idempotency_max"],
                float(np.max(np.abs(projector @ projector - projector)))
            )
            validation["rank_error_max"] = max(
                validation["rank_error_max"], abs(float(np.trace(projector).real) - RANK)
            )

        if step == 0:
            first_h, first_p_inst, first_p_cont = h.copy(), p_inst.copy(), p_cont.copy()

        # Physical-wall chirality reference from the overlap-continued Hamiltonian modes.
        x_local = np.repeat(np.arange(NX), 2)
        wall_weight = np.stack([
            np.sum(np.abs(continued_vecs[:, circular_distance(x_local, center, NX) <= PRIMARY_HALF_WIDTH, :]) ** 2, axis=1)
            for center in DW_INTERVAL
        ], axis=2)
        physical["phi"].append(float(phi))
        physical["evals"].append(continued_evals.copy())
        physical["wall"].append(wall_weight.copy())
        physical["continuation"].append(continuation_diag)

        wall_masks = [circular_distance(full_x, center, NX) <= PRIMARY_HALF_WIDTH for center in DW_INTERVAL]
        wall_charge["phi"].append(float(phi))
        wall_charge["instantaneous"].append([float(np.trace(p_inst[np.ix_(m, m)]).real) for m in wall_masks])
        wall_charge["continued"].append([float(np.trace(p_cont[np.ix_(m, m)]).real) for m in wall_masks])
        diag_inst = np.real(np.diag(p_inst))
        diag_cont = np.real(np.diag(p_cont))
        wall_charge["basin_instantaneous"].append(basin_weights @ diag_inst)
        wall_charge["basin_continued"].append(basin_weights @ diag_cont)
        wall_charge["density_x_instantaneous"].append(diag_inst.reshape(NY, NX, 2).sum(axis=(0, 2)))
        wall_charge["density_x_continued"].append(diag_cont.reshape(NY, NX, 2).sum(axis=(0, 2)))

        for (family, _), tracker in trackers.items():
            tracker.update(phi, p_inst if family == "instantaneous" else p_cont)

        # Compare exact diagonal gauge transforms at representative points.
        if step in {0, len(phi_values)//2, len(phi_values)-1}:
            e_uniform = np.linalg.eigvalsh(h)
            for seam in (0, 1):
                phase = phase_to_seam(phi, seam)
                h_seam = apply_diagonal_gauge(h, phase)
                validation["seam_spectrum_max"] = max(
                    validation["seam_spectrum_max"],
                    float(np.max(np.abs(e_uniform - np.linalg.eigvalsh(h_seam))))
                )
                for spec in CUT_SPECS.values():
                    idx = spec["indices"]
                    c0 = p_inst[np.ix_(idx, idx)]
                    cs = apply_diagonal_gauge(p_inst, phase)[np.ix_(idx, idx)]
                    validation["seam_entanglement_max"] = max(
                        validation["seam_entanglement_max"],
                        float(np.max(np.abs(np.linalg.eigvalsh(c0) - np.linalg.eigvalsh(cs))))
                    )

    last_h, last_p_inst, last_p_cont = h, p_inst, p_cont
    phase = large_gauge_phase(direction)
    validation["h_2pi_closure"] = float(np.max(np.abs(last_h - apply_diagonal_gauge(first_h, phase))))
    validation["instantaneous_2pi_closure"] = float(
        np.max(np.abs(last_p_inst - apply_diagonal_gauge(first_p_inst, phase)))
    )
    # Continued-state monodromy is reported, not required to vanish.
    validation["continued_monodromy"] = float(
        np.linalg.norm(last_p_cont - apply_diagonal_gauge(first_p_cont, phase), ord="fro")
        / max(1.0, np.linalg.norm(first_p_cont, ord="fro"))
    )
    physical = {
        "phi": np.asarray(physical["phi"]),
        "evals": np.stack(physical["evals"]),
        "wall": np.stack(physical["wall"]),
        "continuation": physical["continuation"],
    }
    wall_charge = {
        "phi": np.asarray(wall_charge["phi"]),
        "instantaneous": np.asarray(wall_charge["instantaneous"]),
        "continued": np.asarray(wall_charge["continued"]),
        "basin_instantaneous": np.asarray(wall_charge["basin_instantaneous"]),
        "basin_continued": np.asarray(wall_charge["basin_continued"]),
        "density_x_instantaneous": np.asarray(wall_charge["density_x_instantaneous"]),
        "density_x_continued": np.asarray(wall_charge["density_x_continued"]),
    }
    return {
        "dw_truncation": bool(dw_truncation), "direction": int(direction),
        "global_grid": global_grid, "refined_grid": refined, "phi": phi_values,
        "grid_indices": grid_index_sets(phi_values, global_grid, refined),
        "trackers": trackers, "physical": physical,
        "wall_charge": wall_charge, "validation": validation,
    }

def acceptance_table(cases):
    rows = []
    for key, case in cases.items():
        v = case["validation"]
        for diagnostic in (
            "hermiticity_max", "block_offdiag_max", "block_reconstruction_max",
            "projector_idempotency_max", "rank_error_max", "seam_spectrum_max",
            "seam_entanglement_max", "h_2pi_closure", "instantaneous_2pi_closure",
        ):
            tolerance = PROJECTOR_CLOSURE_TOL if diagnostic == "instantaneous_2pi_closure" else NUMERICAL_TOL
            rows.append({
                "case": str(key), "diagnostic": diagnostic, "value": v[diagnostic],
                "tolerance": tolerance, "pass": bool(v[diagnostic] < tolerance),
            })
        rows.append({
            "case": str(key), "diagnostic": "continued_monodromy",
            "value": v["continued_monodromy"], "tolerance": np.nan, "pass": True,
        })
    return pd.DataFrame(rows)

## Execute the production benchmark

In [ ]:
_started = time.perf_counter()
BENCHMARK_RESULTS = {}
for _truncation in DW_TRUNCATIONS:
    for _direction in TWIST_DIRECTIONS:
        BENCHMARK_RESULTS[(_truncation, _direction)] = run_case(_truncation, _direction)
RUNTIME_SECONDS = time.perf_counter() - _started

ACCEPTANCE = acceptance_table(BENCHMARK_RESULTS)
GAUGE_SUMMARY = ACCEPTANCE.loc[ACCEPTANCE["diagnostic"].isin(
    ["seam_spectrum_max", "seam_entanglement_max", "h_2pi_closure", "instantaneous_2pi_closure"]
)].copy()
display(ACCEPTANCE)
if not ACCEPTANCE.loc[ACCEPTANCE["diagnostic"] != "continued_monodromy", "pass"].all():
    failed = ACCEPTANCE.loc[~ACCEPTANCE["pass"]]
    raise AssertionError("Numerical acceptance checks failed:\n" + failed.to_string(index=False))
print(f"Completed in {RUNTIME_SECONDS/60:.2f} min.")

## Exact-state wall-resolved charge pump

The instantaneous family reselects the exact rank-(N_xN_y) ground state at every twist and serves as the closed-loop control. The continued family starts from that exact ground state and retains its block-resolved occupations while the eigenvectors are overlap-continued through the cycle. Two complementary Voronoi basins centered on the walls cover the full lattice, so their charge residual tests number conservation exactly. Narrow wall windows and the final (x)-resolved excess density test whether the transported charge actually accumulates at the interfaces.

In [ ]:
def exact_charge_pump_table(cases):
    rows = []
    for (truncation, direction), case in cases.items():
        charge = case["wall_charge"]
        for family in ("instantaneous", "continued"):
            basin = charge[f"basin_{family}"]
            narrow = charge[family]
            density_x = charge[f"density_x_{family}"]
            delta_basin = basin[-1] - basin[0]
            delta_narrow = narrow[-1] - narrow[0]
            delta_density = density_x[-1] - density_x[0]
            pump = 0.5 * (delta_basin[1] - delta_basin[0])
            residual = float(delta_basin.sum())
            wall_x_mask = np.logical_or(
                circular_distance(np.arange(NX), DW_INTERVAL[0], NX) <= PRIMARY_HALF_WIDTH,
                circular_distance(np.arange(NX), DW_INTERVAL[1], NX) <= PRIMARY_HALF_WIDTH,
            )
            absolute_excess = float(np.sum(np.abs(delta_density)))
            localization_fraction = (
                float(np.sum(np.abs(delta_density[wall_x_mask]))) / absolute_excess
                if absolute_excess > 1e-14 else np.nan
            )
            rows.append({
                "dw_truncation": truncation, "direction": direction,
                "state_family": family,
                "delta_Q_left_basin": float(delta_basin[0]),
                "delta_Q_right_basin": float(delta_basin[1]),
                "q_pump": float(pump),
                "charge_residual": residual,
                "delta_Q_left_narrow": float(delta_narrow[0]),
                "delta_Q_right_narrow": float(delta_narrow[1]),
                "wall_localization_fraction": localization_fraction,
                "max_total_charge_drift": float(np.max(np.abs(basin.sum(axis=1) - basin.sum(axis=1)[0]))),
            })
    table = pd.DataFrame(rows)
    table["twist_reversal_error"] = np.nan
    for truncation in DW_TRUNCATIONS:
        for family in ("instantaneous", "continued"):
            pick = (table["dw_truncation"] == truncation) & (table["state_family"] == family)
            qplus = table.loc[pick & (table["direction"] == 1), "q_pump"]
            qminus = table.loc[pick & (table["direction"] == -1), "q_pump"]
            if len(qplus) and len(qminus):
                table.loc[pick, "twist_reversal_error"] = abs(float(qplus.iloc[0] + qminus.iloc[0]))
    return table

CHARGE_PUMP_SUMMARY = exact_charge_pump_table(BENCHMARK_RESULTS)
display(CHARGE_PUMP_SUMMARY)

fig, axes = plt.subplots(2, 2, figsize=(7.05, 5.3), sharex="col", constrained_layout=True)
for col, truncation in enumerate(DW_TRUNCATIONS):
    case = BENCHMARK_RESULTS[(truncation, +1)]
    charge = case["wall_charge"]
    for family, ls in (("instantaneous", ":"), ("continued", "-")):
        basin = charge[f"basin_{family}"]
        delta = basin - basin[0]
        axes[0, col].plot(charge["phi"] / np.pi, delta[:, 0], ls, label=f"{family}, left")
        axes[0, col].plot(charge["phi"] / np.pi, delta[:, 1], ls, label=f"{family}, right", alpha=0.8)
        pump_phi = 0.5 * (delta[:, 1] - delta[:, 0])
        axes[1, col].plot(charge["phi"] / np.pi, pump_phi, ls, label=family)
    axes[0, col].set_title(f"dw_truncation={truncation}")
    axes[0, col].legend(fontsize=6)
    axes[1, col].axhline(0, color="0.3", lw=0.6)
    axes[1, col].set_xlabel(r"$\phi/\pi$")
    axes[1, col].set_ylabel(r"$q_{\rm pump}(\phi)$")
axes[0, 0].set_ylabel(r"$\Delta Q_{\rm basin}$")
save_figure(fig, "exact_state_complementary_basin_charge_pump")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.0), sharex=True, sharey=True, constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    charge = BENCHMARK_RESULTS[(truncation, +1)]["wall_charge"]
    for family, marker in (("instantaneous", "o"), ("continued", "s")):
        density = charge[f"density_x_{family}"]
        ax.plot(np.arange(NX), density[-1] - density[0], marker=marker, ms=3, lw=0.8, label=family)
    for wall in DW_INTERVAL:
        ax.axvline(wall, color="0.4", lw=0.7, ls=":")
    ax.set_title(f"dw_truncation={truncation}")
    ax.set_xlabel(r"$x$")
    ax.legend(fontsize=7)
axes[0].set_ylabel(r"$\Delta n(x)$ after $2\pi$")
save_figure(fig, "exact_state_final_charge_transfer_profile")
plt.show()

## Crossing tables: individual tracking, label-independent inertia, and physical walls

In [ ]:
def aggregate_flow_tables(cases):
    tracked_rows, inertia_rows, physical_rows = [], [], []
    for (truncation, direction), case in cases.items():
        for (family, cut_name), tracker in case["trackers"].items():
            for count in TRACKED_MODES:
                arr = tracker.arrays(count)
                for grid_name, idx in case["grid_indices"].items():
                    if len(idx) < 2:
                        continue
                    for ref in REFERENCE_LEVELS:
                        for width in WALL_HALF_WIDTHS:
                            crossings = signed_crossings(
                                arr["phi"][idx], arr["eps"][idx], arr["wall"][width][idx], ref
                            )
                            for wall in (0, 1):
                                robust = [r for r in crossings if r["sector"] == wall and not r["ambiguous"]]
                                tracked_rows.append({
                                    "dw_truncation": truncation, "direction": direction,
                                    "state_family": family, "cut": cut_name,
                                    "tracked_modes": count, "grid": grid_name,
                                    "reference": ref, "wall_half_width": width,
                                    "endpoint_half_width": np.nan, "wall": wall,
                                    "signed_flow": int(sum(r["signed_flow"] for r in robust)),
                                    "n_crossings": len(robust),
                                    "n_ambiguous": int(sum(r["ambiguous"] for r in crossings)),
                                    "estimator": "tracked",
                                })
                            if cut_name == "half_y_current":
                                for ewidth in ENDPOINT_HALF_WIDTHS:
                                    sectors = arr["sectors"][(width, ewidth)]
                                    flat_sector = sectors.reshape(sectors.shape[0], sectors.shape[1], 4)
                                    scross = signed_crossings(arr["phi"][idx], arr["eps"][idx], flat_sector[idx], ref)
                                    for sector in range(4):
                                        robust = [r for r in scross if r["sector"] == sector and not r["ambiguous"]]
                                        tracked_rows.append({
                                            "dw_truncation": truncation, "direction": direction,
                                            "state_family": family, "cut": cut_name,
                                            "tracked_modes": count, "grid": grid_name,
                                            "reference": ref, "wall_half_width": width,
                                            "endpoint_half_width": ewidth,
                                            "wall": sector // 2, "endpoint": sector % 2,
                                            "signed_flow": int(sum(r["signed_flow"] for r in robust)),
                                            "n_crossings": len(robust),
                                            "n_ambiguous": int(sum(r["ambiguous"] for r in scross)),
                                            "estimator": "tracked_endpoint",
                                        })
                    if count == PRIMARY_TRACKED_MODES:
                        for ref in REFERENCE_LEVELS:
                            icross = inertia_crossings(arr["phi"][idx], [tracker.inertia[i] for i in idx], ref)
                            inertia_rows.append({
                                "dw_truncation": truncation, "direction": direction,
                                "state_family": family, "cut": cut_name,
                                "tracked_modes": count, "grid": grid_name,
                                "reference": ref,
                                "signed_flow": int(sum(r["signed_flow"] for r in icross if not r["ambiguous"])),
                                "n_intervals": len(icross),
                                "n_ambiguous": int(sum(r["ambiguous"] for r in icross)),
                                "estimator": "inertia",
                            })

        p = case["physical"]
        for grid_name, idx in case["grid_indices"].items():
            flat_e = p["evals"][idx].reshape(len(idx), -1)
            flat_w = p["wall"][idx].reshape(len(idx), -1, 2)
            crossings = signed_crossings(p["phi"][idx], flat_e, flat_w, 0.0)
            for wall in (0, 1):
                robust = [r for r in crossings if r["sector"] == wall and not r["ambiguous"]]
                physical_rows.append({
                    "dw_truncation": truncation, "direction": direction,
                    "grid": grid_name, "wall": wall,
                    "signed_flow": int(sum(r["signed_flow"] for r in robust)),
                    "n_crossings": len(robust),
                    "n_ambiguous": int(sum(r["ambiguous"] for r in crossings)),
                })
    return pd.DataFrame(tracked_rows), pd.DataFrame(inertia_rows), pd.DataFrame(physical_rows)

FLOW_SUMMARY, INERTIA_SUMMARY, PHYSICAL_FLOW_SUMMARY = aggregate_flow_tables(BENCHMARK_RESULTS)

def reversal_checks():
    rows = []
    tables = (
        ("entanglement", FLOW_SUMMARY, ["dw_truncation", "state_family", "cut", "tracked_modes",
          "grid", "reference", "wall_half_width", "endpoint_half_width", "wall", "estimator"]),
        ("inertia", INERTIA_SUMMARY, ["dw_truncation", "state_family", "cut", "tracked_modes",
          "grid", "reference", "estimator"]),
        ("physical", PHYSICAL_FLOW_SUMMARY, ["dw_truncation", "grid", "wall"]),
    )
    for source, frame, keys in tables:
        data = frame.copy()
        for key in keys:
            if data[key].isna().any():
                data[key] = data[key].fillna(-1)
        forward = data.loc[data["direction"] == 1, keys + ["signed_flow"]]
        reverse = data.loc[data["direction"] == -1, keys + ["signed_flow"]]
        merged = forward.merge(reverse, on=keys, suffixes=("_forward", "_reverse"), how="outer")
        for record in merged.to_dict(orient="records"):
            sf, sr = record.get("signed_flow_forward"), record.get("signed_flow_reverse")
            record["source"] = source
            record["pass"] = bool(pd.notna(sf) and pd.notna(sr) and int(sf) == -int(sr))
            rows.append(record)
    return pd.DataFrame(rows)

REVERSAL_CHECKS = reversal_checks()
display(PHYSICAL_FLOW_SUMMARY)
display(FLOW_SUMMARY.query(
    "estimator == 'tracked' and tracked_modes == @PRIMARY_TRACKED_MODES and reference == 0 and wall_half_width == @PRIMARY_HALF_WIDTH"
).head(30))
display(INERTIA_SUMMARY.head(20))
display(REVERSAL_CHECKS.groupby(["source", "dw_truncation"])["pass"].agg(["sum", "count", "all"]))

## Explicit outcome classification

The classifier is deliberately conservative. A genuine flattened-state result is reported only if all numerical tests pass, twist reversal flips the sign, and every requested grid, tracker count, reference level, localization window, and seam check supports the same wall-resolved integer.

In [ ]:
def classify_results():
    classifications = []
    all_numerical = bool(ACCEPTANCE.loc[ACCEPTANCE["diagnostic"] != "continued_monodromy", "pass"].all())
    for truncation in DW_TRUNCATIONS:
        phys = PHYSICAL_FLOW_SUMMARY.query("dw_truncation == @truncation and grid.str.startswith('global')", engine="python")
        physical_primary = {}
        for direction in TWIST_DIRECTIONS:
            q = phys.query("direction == @direction")
            physical_primary[direction] = tuple(
                int(q.loc[q["wall"] == wall, "signed_flow"].iloc[0]) if np.any(q["wall"] == wall) else 0
                for wall in (0, 1)
            )
        reversal_ok = bool(REVERSAL_CHECKS.loc[REVERSAL_CHECKS["dw_truncation"] == truncation, "pass"].all())
        for family in ("instantaneous", "continued"):
            q = FLOW_SUMMARY.query(
                "dw_truncation == @truncation and state_family == @family and estimator == 'tracked'"
            )
            sensitivity_values = set(q["signed_flow"].astype(int).tolist())
            per_wall_values = {
                wall: set(q.loc[q["wall"] == wall, "signed_flow"].astype(int).tolist())
                for wall in (0, 1)
            }
            grid_artifact = any(len(values) > 1 for values in per_wall_values.values())
            primary = q.query(
                "cut == 'half_y_current' and tracked_modes == @PRIMARY_TRACKED_MODES and "
                "reference == 0 and wall_half_width == @PRIMARY_HALF_WIDTH and grid == 'combined'"
            )
            inertia = INERTIA_SUMMARY.query(
                "dw_truncation == @truncation and state_family == @family and cut == 'half_y_current' and "
                "reference == 0 and grid == 'combined'"
            )
            tracked_total = int(primary["signed_flow"].sum()) if len(primary) else 0
            inertia_total = int(inertia["signed_flow"].sum()) if len(inertia) else 0
            tracking_artifact = tracked_total != inertia_total or bool(primary["n_ambiguous"].sum() if len(primary) else 0)
            endpoint = FLOW_SUMMARY.query(
                "dw_truncation == @truncation and state_family == @family and estimator == 'tracked_endpoint' and "
                "cut == 'half_y_current' and tracked_modes == @PRIMARY_TRACKED_MODES and reference == 0 and "
                "wall_half_width == @PRIMARY_HALF_WIDTH and endpoint_half_width == @PRIMARY_HALF_WIDTH and grid == 'combined'"
            )
            endpoint_cancellation = False
            for wall in (0, 1):
                vals = endpoint.loc[endpoint["wall"] == wall, "signed_flow"].astype(int).tolist()
                endpoint_cancellation |= len(vals) == 2 and sum(vals) == 0 and vals[0] == -vals[1] and vals[0] != 0

            physical_reference = physical_primary.get(+1, (0, 0))
            expected_physical = physical_reference in {(-1, +1), (+1, -1)}
            tested_cuts = q.query(
                "direction == 1 and tracked_modes == @PRIMARY_TRACKED_MODES and reference == 0 and "
                "wall_half_width == @PRIMARY_HALF_WIDTH and grid == 'combined'"
            )
            reproduced = False
            for cut in tested_cuts["cut"].unique():
                qq = tested_cuts.query("cut == @cut")
                candidate = tuple(
                    int(qq.loc[qq["wall"] == wall, "signed_flow"].iloc[0]) if np.any(qq["wall"] == wall) else 0
                    for wall in (0, 1)
                )
                reproduced |= candidate == physical_reference
            cut_mismatch = expected_physical and not reproduced
            genuine = (
                all_numerical and reversal_ok and not grid_artifact and not tracking_artifact
                and not endpoint_cancellation and not cut_mismatch
                and all(len(v) == 1 for v in per_wall_values.values())
            )
            labels = []
            if grid_artifact:
                labels.append("grid artifact / sensitivity failure")
            if tracking_artifact:
                labels.append("tracking artifact or ambiguous multi-crossing interval")
            if endpoint_cancellation:
                labels.append("endpoint cancellation")
            if cut_mismatch:
                labels.append("cut mismatch")
            if genuine:
                labels.append("genuine flattened-state result")
            if not labels:
                labels.append("inconclusive sensitivity result")
            classifications.append({
                "dw_truncation": truncation, "state_family": family,
                "physical_wall_flow_forward": physical_reference,
                "twist_reversal_ok": reversal_ok,
                "grid_artifact": grid_artifact,
                "tracking_artifact": tracking_artifact,
                "endpoint_cancellation": endpoint_cancellation,
                "cut_mismatch": cut_mismatch,
                "genuine_flattened_state": genuine,
                "classification": "; ".join(labels),
            })
    return pd.DataFrame(classifications)

CLASSIFICATION = classify_results()
display(CLASSIFICATION)

## Physical flattened-H wall-energy chirality reference

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.0), sharex=True, sharey=True, constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    p = BENCHMARK_RESULTS[(truncation, +1)]["physical"]
    i0 = int(np.argmin(np.abs(p["phi"] - PHI_OFFSET)))
    ky = 2 * np.pi * np.fft.fftfreq(NY) + p["phi"][i0] / NY
    order = np.argsort(ky)
    for wall, color in enumerate(("tab:blue", "tab:orange")):
        for mode in range(ORBITALS_PER_Y):
            weight = p["wall"][i0, :, mode, wall]
            keep = (weight[order] > 0.20) & (np.abs(p["evals"][i0, order, mode]) < 1.0)
            ax.plot(ky[order][keep] / np.pi, p["evals"][i0, order, mode][keep],
                    ".", ms=2.0, color=color, alpha=0.55)
    ax.axhline(0, color="0.2", lw=0.7)
    ax.set_title(f"dw_truncation={truncation}")
    ax.set_xlabel(r"$k_y/\pi$")
axes[0].set_ylabel("flattened-H wall energy")
fig.suptitle("Physical-wall dispersion / chirality reference")
save_figure(fig, "physical_wall_energy_vs_ky")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.0), sharex=True, sharey=True, constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    case = BENCHMARK_RESULTS[(truncation, +1)]
    p = case["physical"]
    e = p["evals"].reshape(len(p["phi"]), -1)
    w = p["wall"].reshape(len(p["phi"]), -1, 2)
    for wall, color in enumerate(("tab:blue", "tab:orange")):
        mask = np.argmax(w, axis=2) == wall
        for mode in range(e.shape[1]):
            keep = mask[:, mode] & (w[:, mode, wall] > 0.20) & (np.abs(e[:, mode]) < 1.0)
            ax.plot(p["phi"][keep] / np.pi, e[keep, mode], ".", ms=1.2, color=color, alpha=0.45)
    ax.axhline(0, color="0.2", lw=0.7)
    ax.set_title(f"dw_truncation={truncation}")
    ax.set_xlabel(r"$\phi/\pi$")
axes[0].set_ylabel("flattened-H wall energy")
fig.suptitle("Physical-wall twist flow")
save_figure(fig, "physical_wall_energy_vs_twist")
plt.show()

## Full-loop and near-$\pi$ entanglement spectra

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.05, 5.4), sharex="col", sharey=True, constrained_layout=True)
for col, truncation in enumerate(DW_TRUNCATIONS):
    case = BENCHMARK_RESULTS[(truncation, +1)]
    for row, family in enumerate(("instantaneous", "continued")):
        arr = case["trackers"][(family, "half_y_current")].arrays(PRIMARY_TRACKED_MODES)
        axes[row, col].plot(arr["phi"] / np.pi, arr["eps"], lw=0.55, alpha=0.75)
        axes[row, col].axhline(0, color="k", lw=0.6)
        axes[row, col].axvline(1, color="0.5", lw=0.6, ls=":")
        axes[row, col].set_ylim(-1.0, 1.0)
        axes[row, col].set_title(f"{family}; trunc={truncation}")
        axes[row, col].set_xlabel(r"$\phi/\pi$")
        if col == 0:
            axes[row, col].set_ylabel(r"$\epsilon$")
save_figure(fig, "entanglement_spectrum_full_loop")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.0), sharex=True, sharey=True, constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    case = BENCHMARK_RESULTS[(truncation, +1)]
    for family, color in (("instantaneous", "tab:blue"), ("continued", "tab:orange")):
        arr = case["trackers"][(family, "half_y_current")].arrays(PRIMARY_TRACKED_MODES)
        keep = (arr["phi"] / np.pi >= 0.68) & (arr["phi"] / np.pi <= 1.07)
        ax.plot(arr["phi"][keep] / np.pi, arr["eps"][keep], color=color, lw=0.55, alpha=0.55)
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(1, color="0.5", lw=0.6, ls=":")
    ax.set_title(f"trunc={truncation}")
    ax.set_xlabel(r"$\phi/\pi$")
axes[0].set_ylabel(r"$\epsilon$")
save_figure(fig, "entanglement_spectrum_near_pi")
plt.show()

## Assignment quality, principal angles, and selection gaps

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.05, 5.4), sharex=True, constrained_layout=True)
for col, truncation in enumerate(DW_TRUNCATIONS):
    case = BENCHMARK_RESULTS[(truncation, +1)]
    arr = case["trackers"][("continued", "half_y_current")].arrays(PRIMARY_TRACKED_MODES)
    axes[0, col].plot(arr["phi"] / np.pi, arr["assigned"].min(axis=1), label="assigned min")
    axes[0, col].plot(arr["phi"] / np.pi, arr["competing"].max(axis=1), label="competing max")
    axes[0, col].plot(arr["phi"] / np.pi, arr["principal"].min(axis=1), label="principal cos min")
    axes[0, col].set_ylim(0, 1.03)
    axes[0, col].set_title(f"trunc={truncation}")
    axes[0, col].legend(fontsize=7)
    axes[1, col].semilogy(arr["phi"] / np.pi, np.maximum(arr["selection_gap"], 1e-16))
    axes[1, col].set_xlabel(r"$\phi/\pi$")
    axes[1, col].set_ylabel("central-mode selection gap")
axes[0, 0].set_ylabel("overlap / principal cosine")
save_figure(fig, "tracking_overlap_principal_angles_selection_gaps")
plt.show()

## Four half-cut sectors and instantaneous/continued wall charge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.1), constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    case = BENCHMARK_RESULTS[(truncation, +1)]
    arr = case["trackers"][("continued", "half_y_current")].arrays(PRIMARY_TRACKED_MODES)
    ipi = int(np.argmin(np.abs(arr["phi"] - np.pi)))
    sector = arr["sectors"][(PRIMARY_HALF_WIDTH, PRIMARY_HALF_WIDTH)][ipi]
    central = np.argsort(np.abs(arr["eps"][ipi]))[:min(8, PRIMARY_TRACKED_MODES)]
    image = ax.imshow(sector[central].reshape(len(central), 4), aspect="auto", cmap="viridis")
    ax.set_xticks(range(4), ["L/e0", "L/e1", "R/e0", "R/e1"])
    ax.set_yticks(range(len(central)), [str(i) for i in central])
    ax.set_title(f"near-pi sector weights; trunc={truncation}")
    fig.colorbar(image, ax=ax, fraction=0.05)
save_figure(fig, "near_pi_endpoint_wall_sector_weights")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.0), sharex=True, constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    charge = BENCHMARK_RESULTS[(truncation, +1)]["wall_charge"]
    for family, ls in (("instantaneous", "-"), ("continued", "--")):
        delta = charge[family] - charge[family][0]
        ax.plot(charge["phi"] / np.pi, delta[:, 0], ls, label=f"{family}, left")
        ax.plot(charge["phi"] / np.pi, delta[:, 1], ls, label=f"{family}, right", alpha=0.75)
    ax.set_title(f"trunc={truncation}")
    ax.set_xlabel(r"$\phi/\pi$")
    ax.legend(fontsize=6)
axes[0].set_ylabel(r"$\Delta Q_{\rm wall}$")
save_figure(fig, "instantaneous_vs_continued_wall_charge")
plt.show()

## Grid, tracker, cut, direction, and reference-level flow summary

In [ ]:
_plot_flow = FLOW_SUMMARY.query(
    "estimator == 'tracked' and wall_half_width == @PRIMARY_HALF_WIDTH"
).copy()
_plot_flow["label"] = (
    _plot_flow["cut"].astype(str) + "\nK=" + _plot_flow["tracked_modes"].astype(str)
    + "\nref=" + _plot_flow["reference"].map(lambda x: f"{x:+.3f}")
)
fig, axes = plt.subplots(1, 2, figsize=(7.05, 3.4), sharey=True, constrained_layout=True)
for ax, truncation in zip(axes, DW_TRUNCATIONS):
    q = _plot_flow.query(
        "dw_truncation == @truncation and direction == 1 and state_family == 'continued' and grid == 'combined'"
    )
    pivot = q.pivot_table(index="label", columns="wall", values="signed_flow", aggfunc="first").fillna(0)
    image = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="coolwarm",
                      vmin=-max(1, np.max(np.abs(pivot.to_numpy()))),
                      vmax=max(1, np.max(np.abs(pivot.to_numpy()))))
    ax.set_yticks(range(len(pivot)), pivot.index, fontsize=5)
    ax.set_xticks([0, 1], ["left wall", "right wall"])
    ax.set_title(f"trunc={truncation}")
    fig.colorbar(image, ax=ax, fraction=0.05)
save_figure(fig, "flow_robustness_summary")
plt.show()
display(CLASSIFICATION)

## Compact output bundle

Only compact spectra, localization weights, assignments, crossing/diagnostic tables, and figures are written. Dense Hamiltonians and full eigenvector histories are never serialized.

In [ ]:
def compact_payload(cases):
    payload = {}
    for (truncation, direction), case in cases.items():
        tag = f"trunc_{int(truncation)}_dir_{direction:+d}".replace("+", "p").replace("-", "m")
        payload[f"{tag}__phi"] = case["phi"]
        payload[f"{tag}__physical_evals"] = case["physical"]["evals"].astype(np.float32)
        payload[f"{tag}__physical_wall_weights"] = case["physical"]["wall"].astype(np.float32)
        for family in ("instantaneous", "continued"):
            payload[f"{tag}__{family}__basin_charge"] = case["wall_charge"][f"basin_{family}"].astype(np.float64)
            payload[f"{tag}__{family}__density_x"] = case["wall_charge"][f"density_x_{family}"].astype(np.float32)
        for (family, cut_name), tracker in case["trackers"].items():
            for count in TRACKED_MODES:
                arr = tracker.arrays(count)
                prefix = f"{tag}__{family}__{cut_name}__K{count}"
                payload[f"{prefix}__eps"] = arr["eps"].astype(np.float32)
                payload[f"{prefix}__assigned"] = arr["assigned"].astype(np.float32)
                payload[f"{prefix}__competing"] = arr["competing"].astype(np.float32)
                payload[f"{prefix}__principal"] = arr["principal"].astype(np.float32)
                payload[f"{prefix}__selection_gap"] = arr["selection_gap"].astype(np.float32)
                for width, values in arr["wall"].items():
                    payload[f"{prefix}__wall_w{width}"] = values.astype(np.float32)
                for (wwidth, ewidth), values in arr["sectors"].items():
                    payload[f"{prefix}__sector_w{wwidth}_e{ewidth}"] = values.astype(np.float32)
    return payload

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FLOW_SUMMARY.to_csv(OUTPUT_DIR / "tracked_crossing_summary.csv", index=False)
    INERTIA_SUMMARY.to_csv(OUTPUT_DIR / "inertia_crossing_summary.csv", index=False)
    PHYSICAL_FLOW_SUMMARY.to_csv(OUTPUT_DIR / "physical_wall_flow_summary.csv", index=False)
    ACCEPTANCE.to_csv(OUTPUT_DIR / "numerical_acceptance.csv", index=False)
    GAUGE_SUMMARY.to_csv(OUTPUT_DIR / "gauge_and_large_gauge_closure.csv", index=False)
    REVERSAL_CHECKS.to_csv(OUTPUT_DIR / "twist_reversal_checks.csv", index=False)
    CLASSIFICATION.to_csv(OUTPUT_DIR / "outcome_classification.csv", index=False)
    CHARGE_PUMP_SUMMARY.to_csv(OUTPUT_DIR / "exact_state_wall_charge_pump.csv", index=False)
    np.savez_compressed(OUTPUT_DIR / "compact_spectra_weights_assignments.npz", **compact_payload(BENCHMARK_RESULTS))
    manifest = {
        "run_id": RUN_ID, "runtime_seconds": RUNTIME_SECONDS,
        "configuration": CONFIG, "source_sha256": SOURCE_HASH,
        "continued_monodromy": {
            str(key): case["validation"]["continued_monodromy"]
            for key, case in BENCHMARK_RESULTS.items()
        },
        "exact_state_charge_pump": json.loads(CHARGE_PUMP_SUMMARY.to_json(orient="records")),
        "note": "Estimator benchmark for exact flattened states; not monitored-trajectory evidence.",
    }
    (OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=list))
    print(f"Saved compact benchmark bundle to {OUTPUT_DIR}")

## Interpretation checklist

Use the generated classification and plots together:

- **Grid artifact:** the near-\(\pi\) lobe or its count changes between nested/refined meshes.
- **Tracking artifact:** individual-mode flow disagrees with polar/subspace diagnostics or label-independent inertia; ambiguous multi-crossing intervals are retained as ambiguous.
- **Endpoint cancellation:** opposite endpoint-resolved flows sum to zero on the same physical wall.
- **Cut mismatch:** physical flattened-H flow is \((-1,+1)\) up to orientation, but none of the half-\(y\) or full-\(y\) cuts reproduces it robustly.
- **Genuine flattened-state result:** all numerical, gauge, direction, mesh, tracker-count, reference-level, wall-window, endpoint-window, and cut checks agree.

The continued projector is not required to close. Its large-gauge-relative monodromy is part of the output and should be interpreted alongside the instantaneous \(2\pi\) closure.